In [1]:
import pandas as pd
import shutil
from pathlib import Path
from sklearn.model_selection import GroupKFold
from tqdm import tqdm

# ======== 參數設定 ========
# 支援多個輸入目錄
IN_DIRS = ["data", "aug_data", "other", "other_app"]
OUT_ROOT = Path("K_Fold")       # 最終輸出目錄
WINDOW = 960
STRIDE = 480
K_FOLDS = 5

def main():
    file_meta = []

    # 1. 遍歷所有指定的資料夾，取得 CSV 列表並標記 Group 與 Label
    for folder_name in IN_DIRS:
        dir_path = Path(folder_name)
        
        if not dir_path.exists():
            print(f"[Warning] 找不到目錄 {folder_name}，將略過此資料夾。")
            continue
            
        raw_files = sorted(list(dir_path.glob("*.csv")))
        
        for p in raw_files:
            # 確保檔名前面的蒐集日期相同者不可分開：取第一個底線前的內容作為 Group ID
            group_id = p.name.split("_")[0]
            
            # 依據資料來源資料夾與檔名決定標籤 (Label)
            if folder_name in ["other", "other_app"]:
                label = "other"
            else:  # folder_name 為 "data" 或 "aug_data"
                if "notTired" in p.name:
                    label = "notTired"
                else:
                    label = "Tired"
                    
            file_meta.append({"path": p, "label": label, "group": group_id, "source_dir": folder_name})

    if not file_meta:
        print("[Error] 所有資料夾內皆找不到 CSV 檔案，請檢查路徑。")
        return

    df_meta = pd.DataFrame(file_meta)
    print(f"共讀取 {len(df_meta)} 個檔案，包含 {df_meta['group'].nunique()} 個獨立 Group。")
    print("各類別檔案數量：\n", df_meta["label"].value_counts().to_string())

    # 2. 準備交叉驗證 (GroupKFold)
    gkf = GroupKFold(n_splits=K_FOLDS)
    
    # 清空輸出目錄
    if OUT_ROOT.exists(): shutil.rmtree(OUT_ROOT)

    # 3. 開始執行 Fold 循環
    for fold, (train_idx, val_idx) in enumerate(gkf.split(df_meta, groups=df_meta["group"]), start=1):
        print(f"\n=== 處理 Fold {fold}/{K_FOLDS} ===")
        
        # 定義這個 Fold 的任務內容
        job_config = {
            "train": df_meta.iloc[train_idx],
            "val":   df_meta.iloc[val_idx]  # 此處命名改為 val 以利區分
        }

        for split_name, split_df in job_config.items():
            for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"Processing {split_name}", leave=False):
                # 讀取並切割數據 (僅保留數值欄位)
                df_raw = pd.read_csv(row["path"]).select_dtypes(include=['number'])
                base_name = row["path"].stem
                
                # 滑動視窗切割與儲存
                for start in range(0, len(df_raw) - WINDOW + 1, STRIDE):
                    chunk = df_raw.iloc[start : start + WINDOW]
                    out_filename = f"{base_name}_s{start:06d}.csv"
                    
                    # 建立格式化目錄：K_Fold/fold_1/train/{label}/xxx.csv
                    target_dir = OUT_ROOT / f"fold_{fold}" / split_name / row["label"]
                    target_dir.mkdir(parents=True, exist_ok=True)
                    
                    chunk.to_csv(target_dir / out_filename, index=False)

    print(f"\n任務完成！輸出資料夾: {OUT_ROOT.resolve()}")

if __name__ == "__main__":
    main()

共讀取 263 個檔案，包含 90 個獨立 Group。
各類別檔案數量：
 label
notTired    100
other        91
Tired        72

=== 處理 Fold 1/5 ===



=== 處理 Fold 2/5 ===



=== 處理 Fold 3/5 ===



=== 處理 Fold 4/5 ===



=== 處理 Fold 5/5 ===



任務完成！輸出資料夾: /home/mluser/114_NCUT_A100/data_v56/K_Fold
